In [1]:
import requests
import json
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
APP_ID = os.getenv("ADZUNA_APP_ID")
APP_KEY = os.getenv("ADZUNA_APP_KEY")


In [2]:
response = requests.get(
    "https://api.adzuna.com/v1/api/jobs/gb/search/1",
    params={
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 50,
        "what": "data engineer"
    }
)

response.raise_for_status()  # Raises an exception for HTTP errors

data = response.json()

# Save it to inspect without hitting the API again
with open("../data/raw/sample_response.json", "w") as f:
    json.dump(data, f, indent=2)

In [4]:
# Top level keys
print(data.keys())

dict_keys(['results', '__CLASS__', 'count', 'mean'])


In [3]:
df = pd.json_normalize(data["results"])

# First five job fields
df.head()

,title,__CLASS__,adref,salary_is_predicted,salary_min,redirect_url,salary_max,description,created,id,...,category.label,company.display_name,company.__CLASS__,location.display_name,location.area,location.__CLASS__,longitude,contract_type,latitude,contract_time
0,Data Engineer,Adzuna::API::Response::Job,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgxMzkxMjE3OCIsI...,0,55000.00,https://www.adzuna.co.uk/jobs/land/ad/58139121...,55000.00,Data Engineer Location: Belfast (Hybrid) Eligi...,2026-07-24T15:23:07Z,5813912178,...,IT Jobs,Ocho,Adzuna::API::Response::Company,"Belfast, Northern Ireland","[UK, Northern Ireland, Belfast]",Adzuna::API::Response::Location,NaN,NaN,NaN,NaN
1,Data Engineer,Adzuna::API::Response::Job,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgyNjgyMTcyMCIsI...,1,54838.96,https://www.adzuna.co.uk/jobs/land/ad/58268217...,54838.96,Company description We believe in the power of...,2026-08-03T21:20:43Z,5826821720,...,IT Jobs,PA Consulting,Adzuna::API::Response::Company,"Belfast, Northern Ireland","[UK, Northern Ireland, Belfast]",Adzuna::API::Response::Location,NaN,NaN,NaN,NaN
2,Data Engineer,Adzuna::API::Response::Job,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgxMjY0NTY1NCIsI...,1,53868.62,https://www.adzuna.co.uk/jobs/land/ad/58126456...,53868.62,Company description We believe in the power of...,2026-07-23T21:03:51Z,5812645654,...,IT Jobs,PA Consulting,Adzuna::API::Response::Company,"Belfast, Northern Ireland","[UK, Northern Ireland, Belfast]",Adzuna::API::Response::Location,NaN,NaN,NaN,NaN
3,Data Engineer,Adzuna::API::Response::Job,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgzNzAyNDQ4MCIsI...,0,50000.00,https://www.adzuna.co.uk/jobs/land/ad/58370244...,50000.00,Data Engineer Newcastle (Hybrid) Permanent Com...,2026-08-11T17:19:14Z,5837024480,...,IT Jobs,Anson Mccade,Adzuna::API::Response::Company,"Newcastle Upon Tyne, Tyne & Wear","[UK, North East England, Tyne & Wear, Newcastl...",Adzuna::API::Response::Location,-1.746207,permanent,55.028274,NaN
4,Data Engineer,Adzuna::API::Response::Job,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgwNDkwNzQ0NiIsI...,0,100000.00,https://www.adzuna.co.uk/jobs/land/ad/58049074...,100000.00,Azure SQL DBA / Data Platform Engineer | Belfa...,2026-07-17T14:24:31Z,5804907446,...,IT Jobs,MCS Group,Adzuna::API::Response::Company,"Belfast, Northern Ireland","[UK, Northern Ireland, Belfast]",Adzuna::API::Response::Location,NaN,permanent,NaN,NaN


In [11]:
# Inspect job fields
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   description            50 non-null     str    
 1   salary_max             50 non-null     float64
 2   contract_time          20 non-null     str    
 3   latitude               29 non-null     float64
 4   __CLASS__              50 non-null     str    
 5   salary_is_predicted    50 non-null     str    
 6   salary_min             50 non-null     float64
 7   id                     50 non-null     str    
 8   adref                  50 non-null     str    
 9   title                  50 non-null     str    
 10  longitude              29 non-null     float64
 11  redirect_url           50 non-null     str    
 12  created                50 non-null     str    
 13  location.__CLASS__     50 non-null     str    
 14  location.display_name  50 non-null     str    
 15  location.area      

In [ ]:
# Check if salary_is_predicted is boolean
df['salary_is_predicted'].unique()

<StringArray>
['1', '0']
Length: 2, dtype: str

In [16]:
# % of records with missing max salary
missing_max_salary = df['salary_max'].isna().mean() * 100

# % of records with missing min salary
missing_min_salary = df['salary_min'].isna().mean() * 100

# % of records where salary_is_predicted = 1
predicted_salary = df['salary_is_predicted'].astype(int).mean() * 100

# duplicate IDs
duplicate_ids = df['id'].duplicated().sum()

print(f"Missing max salary: {missing_max_salary:.1f}%")
print(f"Missing min salary: {missing_min_salary:.1f}%")
print(f"Predicted salary: {predicted_salary:.1f}%")
print(f"Duplicate IDs: {duplicate_ids}")

# title lists
print(df['title'].tolist())

Missing max salary: 0.0%
Missing min salary: 0.0%
Predicted salary: 44.0%
Duplicate IDs: 0
['Chief Engineer, Data Centre Engineering Operations, Data Centre Engineering Operations', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Lead Data Engineer', 'Data Engineer', 'Data Engineer ( AWS )', 'Data Engineer ( Snowflake )', 'Senior Data Engineer', 'Data Engineer - Python', 'Manufacturing Data Engineer', 'Staff Data Engineer', 'Data Engineer', 'Senior Azure Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Data Engineer', 'Senior Data Engineer / Lead Data Engineer', 

In [20]:
# Missing contract_time
missing_contract_time = df['contract_time'].isna().mean() * 100

print(f"Missing contract_time: {missing_contract_time:.1f}%")
print(df['contract_time'].value_counts(dropna=False))

Missing contract_time: 60.0%
contract_time
NaN          30
full_time    18
part_time     2
Name: count, dtype: int64


In [21]:
# Inspect category
print(df['category.label'].value_counts(dropna=False))
print(df['category.tag'].value_counts(dropna=False))

category.label
IT Jobs             48
Engineering Jobs     2
Name: count, dtype: int64
category.tag
it-jobs             48
engineering-jobs     2
Name: count, dtype: int64


In [4]:
# Inpect contract_type
missing_contract_type = df['contract_type'].isna().mean() * 100

print(f"Missing contract_type: {missing_contract_type:.1f}%")
print(df['contract_type'].value_counts(dropna=False))

Missing contract_type: 70.0%
contract_type
NaN          35
permanent    13
contract      2
Name: count, dtype: int64


In [9]:
# How many descriptions end in truncation?
truncated = df["description"].str.strip().str.endswith("…").sum()
print(f"Truncated descriptions: {truncated}/{len(df)}")

# Spot-check a few descriptions for boilerplate-only pattern
for desc in df["description"].sample(5):
    print(desc[:500])
    print("---")

Truncated descriptions: 50/50
Your New Role Short term contract for 4x experienced Data Cabling Engineers to join a commercial installation project. This role is ideal for engineers who are confident installing, terminating and testing Cat 6A structured cabling and are comfortable working at height using powered access equipment. Your Responsibilities Installation of Cat 6A structured cabling.Termination of data outlets, patch panels and cabinets.Testing and certification of installed cabling.Working from drawings and cable…
---
At Data Intellect it has never been just about data or technology, they are our tools. It's about human intellect, collaboration and providing solutions for the most complex of challenges. We do this by living the [DI] code: We are Problem Solvers who are Humble, possess a Can-do Attitude with a focus on Togetherness. "We are not big on egos, but we're not for the faint-hearted either" - Steve Turner, CEO What you'll be doing: Implement, configure and evolve Sa

In [ ]:
# Average description length
df["description"].str.len().describe()

count     50.0
mean     500.0
std        0.0
min      500.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: description, dtype: float64

In [5]:
# Check which fields have null/missing values
df.isna().sum()

description               0
adref                     0
title                     0
redirect_url              0
salary_is_predicted       0
salary_min                0
longitude                28
__CLASS__                 0
id                        0
created                   0
contract_type            20
latitude                 28
salary_max                0
company.__CLASS__         0
company.display_name      0
location.display_name     0
location.__CLASS__        0
location.area             0
category.tag              0
category.label            0
category.__CLASS__        0
contract_time            27
dtype: int64